# Phase 5 - Model Improvement & Optimization

## Objective

The objective of this phase is to improve the performance of regression models using different optimization techniques.

### Optimization Techniques

- Feature Engineering
- Feature Selection
- Data Transformation
- Outlier Treatment
- Missing Value Handling
- Cross Validation
- Hyperparameter Tuning
- Multicollinearity Analysis (VIF)

### Expected Outcome

- Improve prediction accuracy
- Reduce overfitting
- Compare model performance before and after optimization
- Select the best optimized model

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.stats import skew
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor



import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
!{sys.executable} -m pip install statsmodels

In [3]:
# Load Dataset
df=pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_08/Notebooks/dataset/house_prices_cleaned.csv")

df.head()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,HouseAge,TotalBathrooms,TotalSF,TotalPorchArea,RemodeledHouse
0,1,4.110874,4.189655,9.042040,7,5,2003,2003,5.283204,6.561031,...,0,0,0,1,0,5,3.500000,20.257977,4.127134,0
1,2,3.044522,4.394449,9.169623,6,8,1976,1976,0.000000,6.886532,...,0,0,0,1,0,31,2.346574,14.282490,0.000000,0
2,3,4.110874,4.234107,9.328212,7,5,2001,2002,5.093750,6.188264,...,0,0,0,1,0,7,3.500000,20.415959,3.761200,1
3,4,4.262680,4.110874,9.164401,7,5,1915,1970,0.000000,5.379897,...,0,0,0,0,0,91,2.000000,20.127741,9.192991,1
4,5,4.110874,4.442651,9.565284,8,5,2000,2000,5.860786,6.486161,...,0,0,0,1,0,8,3.500000,21.048414,4.442651,0


In [44]:

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

In [45]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [46]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
6666.769874	
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

X_vif = X_train.copy()

vif = pd.DataFrame()
vif["Feature"] = X_vif.columns
vif["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

vif = vif.sort_values("VIF", ascending=False)

vif

,Feature,VIF
6,YearBuilt,inf
18,BsmtHalfBath,inf
19,FullBath,inf
20,HalfBath,inf
17,BsmtFullBath,inf
...,...,...
120,Exterior1st_CBlock,NaN
123,Exterior1st_ImStucc,NaN
134,Exterior2nd_CBlock,NaN
151,ExterCond_Po,NaN


In [8]:
from sklearn.model_selection import cross_val_score

def cv_score(model):
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    return score.mean()

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {

    "Linear Regression":
        LinearRegression(),

    "Decision Tree":
        DecisionTreeRegressor(random_state=42),

    "Random Forest":
        RandomForestRegressor(random_state=42),

    "Gradient Boosting":
        GradientBoostingRegressor(random_state=42),

    "XGBoost":
        XGBRegressor(random_state=42),

    "LightGBM":
        LGBMRegressor(random_state=42),

    "CatBoost":
        CatBoostRegressor(
            random_state=42,
            verbose=0
        )

}

In [10]:
param_grid = {

"Linear Regression":{},

"Decision Tree":{

    "max_depth":[3,5,10,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]

},

"Random Forest":{

    "n_estimators":[100,200,300],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]

},

"Gradient Boosting":{

    "n_estimators":[100,300],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,5]

},

"XGBoost":{

    "n_estimators":[100,300],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,5,7],
    "subsample":[0.8,1]

},

"LightGBM":{

    "n_estimators":[100,300],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[-1,10],
    "num_leaves":[31,50]

},

"CatBoost":{

    "iterations":[200,500,800,1000],
    "depth" : [4,6,8,10],
    "learning_rate":[0.01,0.05,0.1],
    "l2_leaf_reg":[1,3,5]

}

}

In [11]:
from sklearn.model_selection import RandomizedSearchCV

best_models = {}

for name, model in models.items():

    print("="*50)
    print(name)

    if name == "Linear Regression":

        model.fit(X_train, y_train)

        best_models[name] = model

    else:

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=param_grid[name],

            n_iter=10,

            cv=5,

            scoring="r2",

            random_state=42,

            n_jobs=-1

        )

        search.fit(X_train, y_train)

        best_models[name] = search.best_estimator_

        print(search.best_params_)

Linear Regression
Decision Tree
{'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 5}
Random Forest
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 20}
Gradient Boosting
{'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05}
XGBoost
{'subsample': 0.8, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1}
LightGBM
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015929 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3684
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 139
[LightGBM] [Info] Start training from score 171503.298324
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006972 seconds.
You can set `force_

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044829 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3693
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 145
[LightGBM] [Info] Start training from score 171135.127374
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.360227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3686
[LightGBM] [Info] Number of data points in the train set: 896, number of used features: 143
[Lig

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.335159 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3673
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 143
[LightGBM] [Info] Start training from score 171280.436872
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004964 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3673
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 143
[Lig

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.273412 

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.309046 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3684
[LightGBM] [Info] Number of data points in the train set: 89

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042921 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3974
[LightGBM] [Info] Number of data points in the train set: 1119, number of used features: 154
[LightGBM] [Info] Start training from score 171335.218945
{'num_leaves': 31, 'n_estimators': 300, 'max_depth': -1, 'learning_rate': 0.1}
CatBoost
{'learning_rate': 0.05, 'l2_leaf_reg': 1, 'iterations': 800, 'depth': 4}


In [12]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

results = []

for name, model in best_models.items():

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)

    mse = mean_squared_error(y_test, pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, pred)

    adj_r2 = 1 - ((1-r2)*(len(y_test)-1)/(len(y_test)-X_test.shape[1]-1))

    cv = cv_score(model)

    results.append([

        name,

        mae,

        mse,

        rmse,

        r2,

        adj_r2,

        cv

    ])

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002195 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3693
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 145
[LightGBM] [Info] Start training from score 171135.127374
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003113 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3673
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 143
[LightGBM] [Info] Start training from score 171280.436872
[LightGBM] [Warning]


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [13]:
#before HyperTuning
comparison = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_08/Notebooks/dataset/reports/model_comparison.csv")

comparison

,Model,MAE,MSE,RMSE,R2 Score
0,CatBoost,11480.855107,2.794843e+08,16717.783610,0.912134
1,Gradient Boosting,13284.836044,3.576519e+08,18911.687912,0.887560
2,LightGBM,13469.373309,3.833403e+08,19579.078554,0.879484
3,XGBoost,14013.219727,4.109945e+08,20272.999975,0.870790
4,Random Forest,14043.607250,4.354628e+08,20867.744897,0.863097
5,Decision Tree,23677.432143,1.172576e+09,34242.902612,0.631360
6,Linear Regression,16837.207398,1.191990e+09,34525.206485,0.625257


In [14]:
#After Hypertuning
comparison = pd.DataFrame(

    results,

    columns=[

        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "R2 Score",
        "Adjusted R2",
        "CV Score"

    ]   

)

comparison.sort_values(

    by="R2 Score",

    ascending=False,

    inplace=True

)

comparison.reset_index(drop=True, inplace=True)

comparison

,Model,MAE,MSE,RMSE,R2 Score,Adjusted R2,CV Score
0,CatBoost,11548.729812,2.849007e+08,16879.002568,0.910432,0.375260,0.903670
1,XGBoost,12352.547852,3.328377e+08,18243.840824,0.895361,0.270143,0.889148
2,Gradient Boosting,13168.004043,3.600397e+08,18974.712377,0.886809,0.210493,0.889921
3,LightGBM,13694.743054,3.924191e+08,19809.571303,0.876629,0.139491,0.877381
4,Random Forest,13900.934111,4.191185e+08,20472.384926,0.868236,0.080943,0.859284
5,Decision Tree,20793.414061,7.906316e+08,28118.171734,0.751438,-0.733723,0.725241
6,Linear Regression,16837.207398,1.191990e+09,34525.206485,0.625257,-1.613834,0.786586


In [15]:
search.best_estimator_

CatBoostRegressor(depth=4, iterations=800, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', random_state=42, verbose=0)

In [35]:
best_cb_1 = search.best_estimator_

In [21]:

features = [
    "OverallQual",     # Overall material and finish quality
    "GrLivArea",       # Above ground living area
    "GarageCars",      # Garage capacity
    "TotalBsmtSF",     # Total basement area
    "FullBath",        # Number of full bathrooms
    "YearBuilt",       # Construction year
    "1stFlrSF",        # First floor area
    "TotRmsAbvGrd",    # Total rooms above ground
    "Fireplaces",      # Number of fireplaces
    "LotArea"          # Total lot size
]
X = df[features]
y = df["SalePrice"]

In [22]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [30]:
param_grid = {

    "iterations":[200,500,800,1000],
    "depth" : [4,6,8,10],
    "learning_rate":[0.01,0.05,0.1],
    "l2_leaf_reg":[1,3,5]
}
search = RandomizedSearchCV(
    CatBoostRegressor(verbose=0, random_state=42),param_grid,cv=5,scoring='r2',n_jobs=-1
)
search.fit(X_train, y_train)
print(search.best_estimator_)
best_cb = search.best_estimator_
cat_pred = best_cb.predict(X_test)

CatBoostRegressor(depth=6, iterations=500, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', random_state=42, verbose=0)


In [38]:
best_cb = search.best_estimator_
best_cb

CatBoostRegressor(depth=6, iterations=500, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', random_state=42, verbose=0)

In [43]:
# Save Optimized Model
import joblib

joblib.dump(
    best_cb,
    "/home/aximsoft/Documents/AximSoft_EOWA/Week_08/Notebooks/dataset/models/best_model1.pkl"
)

print("Model Saved Successful22ly")

Model Saved Successful22ly
